In [35]:
import fitz  # PyMuPDF
import pandas as pd

In [36]:
pdf_path = "test_sample/Ficha_Ponto_Simplificada_André_Luis.pdf"

In [37]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
doc = fitz.open(pdf_path)
# self.timesheet_data = []

for page in doc:
    tabs = page.find_tables()

    for tab in tabs.tables:
        print(tab.to_pandas())

             Col0      Col1      Col2 SCALA TRANSPORTE E ADMINISTRACAO LTDA. 88.501.093/0001-89     Col4   Col5       Col6             Col7             Col8           Col9  ...            Col16           Col17                                              Col18                  Col19    Col20  Col21                   Col22    Col23  Col24          Col25
0                      None      None  Ficha Ponto Simplificada Período: 21/05/2025 à...            None   None       None             None             None           None  ...             None            None                                               None                   None     None   None                    None     None   None           None
1            None      None      None  Funcionário: ANDRE LUIS DE MORAES DA ROSA CPF:...            None   None       None             None             None           None  ...             None            None                                               None                   None     No

In [38]:
import re
from pandas import DataFrame

# Convert the table to a DataFrame
df_tab = tab.to_pandas()

def reset_column_names(df: DataFrame) -> DataFrame:
    df.columns = [f"Col_{i}" for i in range(df.shape[1])]
    return df

# Define a function to check if a string starts with a date in the format DD/MM/YY
def starts_with_date(val):
    if isinstance(val, str):
        return bool(re.match(r'^\d{2}/\d{2}/\d{2}', val.strip()))
    return False

def drop_unwanted_columns(df: DataFrame) -> DataFrame:
    # Drop all columns that are fully with None
    df = df.dropna(axis=1, how='all')
    return df

# Filter rows where the first column starts with a date
df_tab = reset_column_names(df_tab)
filtered_df = df_tab[df_tab.iloc[:, 0].apply(starts_with_date)].reset_index(drop=True)
df_timesheet = drop_unwanted_columns(filtered_df)


print(df_timesheet)

           Col_0     Col_1  Col_4  Col_5  Col_7  Col_8 Col_10 Col_11 Col_13 Col_14 Col_16 Col_17 Col_19 Col_20 Col_21 Col_22 Col_23 Col_24 Col_25
0   21/05/25 qua  Trabalho  06:24  01:26  08:00  16:59  05:30  12:11  04:48  04:48  01:21  00:42  05:57  03:02  08:59                       03:02
1   22/05/25 qui  Trabalho  10:35  23:42  08:00  11:58  09:09  04:03  07:55  07:55  01:09         02:16  01:42  03:58                       01:42
2   23/05/25 sex  Trabalho  08:05  20:32  08:00  10:56  08:23  05:10  05:46  05:46  01:11  00:20  02:56         02:56                            
3   24/05/25 sáb  Trabalho  08:22  18:02  08:00  08:33  11:50  01:02  07:31  07:31  01:07         00:33         00:33                            
4   25/05/25 dom  Trabalho  07:37  19:07  08:00  10:08  13:35  05:46  04:22  04:22  01:22         02:08         02:08                            
5   26/05/25 seg  Trabalho  05:54  21:17  08:00  14:21  10:47  03:00  11:21  11:21  01:02         06:21         06:21       

In [39]:
column_mapping = {
    'Col_0': 'data',
    'Col_1': 'tipo',
    'Col_4': 'jornada_inicio',
    'Col_5': 'jornada_fim',
    'Col_7': 'jornada_normal',
    'Col_8': 'jornada_diaria',
    'Col_10': 'interjornada',
    'Col_11': 'em_direcao',
    'Col_13': 'total_parado',
    'Col_14': 'sem_direcao',
    'Col_16': 'total_refeicao',
    'Col_17': 'total_repouso',
    'Col_19': 'hora_extra_diaria_diurna',
    'Col_20': 'hora_extra_diaria_noturna',
    'Col_21': 'hora_extra_diaria_total',
    'Col_22': 'hora_extra_dom_fer_diurna',
    'Col_23': 'hora_extra_dom_fer_noturna',
    'Col_24': 'hora_extra_dom_fer_total',
    'Col_25': 'hora_noturna'
}

df_timesheet_renamed = df_timesheet.rename(columns=column_mapping)
print(df_timesheet_renamed)

            data      tipo jornada_inicio jornada_fim jornada_normal jornada_diaria interjornada em_direcao total_parado sem_direcao total_refeicao total_repouso hora_extra_diaria_diurna hora_extra_diaria_noturna hora_extra_diaria_total hora_extra_dom_fer_diurna hora_extra_dom_fer_noturna hora_extra_dom_fer_total hora_noturna
0   21/05/25 qua  Trabalho          06:24       01:26          08:00          16:59        05:30      12:11        04:48       04:48          01:21         00:42                    05:57                     03:02                   08:59                                                                                      03:02
1   22/05/25 qui  Trabalho          10:35       23:42          08:00          11:58        09:09      04:03        07:55       07:55          01:09                                  02:16                     01:42                   03:58                                                                                      01:42
2   23/05/25 sex

In [69]:
# Extract information from the PDF
import re
from datetime import datetime

def extract_timesheet_info(pdf_path):
    doc = fitz.open(pdf_path)
    
    # Variables to store extracted information
    company_name = None
    employee_name = None
    period = None
    daily_data = []
    
    for page in doc:
        # Extract text from the page for company, employee, and period info
        text = page.get_text()
        
        # Extract company name (usually appears at the top)
        # Look for common patterns in Portuguese timesheets
        company_patterns = [
            r'EMPRESA[:\s]*([^\n\r]+)',
            r'RAZÃO SOCIAL[:\s]*([^\n\r]+)',
            r'EMPREGADOR[:\s]*([^\n\r]+)'
        ]
        
        for pattern in company_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not company_name:
                company_name = match.group(1).strip()
                break

        # Extract employee name
        employee_name_match = re.search(r'ANDRE LUIS DE MORAES DA ROSA', text)
        if employee_name_match:
            employee_name = employee_name_match.group(0)
        else:
            # Fallback pattern
            employee_name_match = re.search(r'Funcionário:\s*\n?([A-ZÁÊÇÕ\s]+)', text)
            if employee_name_match:
                employee_name = employee_name_match.group(1).strip()
        
        # Extract period
        period_patterns = [
            r'PERÍODO[:\s]*([^\n\r]+)',
            r'COMPETÊNCIA[:\s]*([^\n\r]+)',
            r'(\d{2}/\d{4})',  # MM/YYYY format
            r'(\d{2}/\d{2}/\d{4}\s*a\s*\d{2}/\d{2}/\d{4})'  # DD/MM/YYYY a DD/MM/YYYY format
        ]
        
        for pattern in period_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not period:
                period = match.group(1).strip()
                break
    
    doc.close()
    
    return company_name, employee_name, period

# Extract the information
company_name, employee_name, period  = extract_timesheet_info(pdf_path)

print("=== EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")  
print(f"Period: {period}")
print(f"\n=== SUMMARY OF DAILY TIMESHEET ===")
print(f"Total days in timesheet: {len(df_timesheet_renamed)}")
print(f"Working days: {len(df_timesheet_renamed[df_timesheet_renamed['tipo'] == 'Trabalho'])}")
print(f"Rest days (DSR/Casa): {len(df_timesheet_renamed[df_timesheet_renamed['tipo'] == 'DSR/Casa'])}")
print(f"Holidays (Feriado): {len(df_timesheet_renamed[df_timesheet_renamed['tipo'] == 'Feriado'])}")


print(f"\n=== SAMPLE OF WORKING DAYS ===")
working_days_sample = df_timesheet_renamed[df_timesheet_renamed['tipo'] == 'Trabalho'].head(10)
print(working_days_sample)

=== EXTRACTED INFORMATION ===
Company/Employer: None
Employee Name: ANDRE LUIS DE MORAES DA ROSA
Period: 21/05/2025 à 20/06/2025

=== SUMMARY OF DAILY TIMESHEET ===
Total days in timesheet: 31
Working days: 25
Rest days (DSR/Casa): 5
Holidays (Feriado): 1

=== SAMPLE OF WORKING DAYS ===
           data      tipo jornada_inicio jornada_fim jornada_normal jornada_diaria interjornada em_direcao total_parado sem_direcao total_refeicao total_repouso hora_extra_diaria_diurna hora_extra_diaria_noturna hora_extra_diaria_total hora_extra_dom_fer_diurna hora_extra_dom_fer_noturna hora_extra_dom_fer_total hora_noturna
0  21/05/25 qua  Trabalho          06:24       01:26          08:00          16:59        05:30      12:11        04:48       04:48          01:21         00:42                    05:57                     03:02                   08:59                                                                                      03:02
1  22/05/25 qui  Trabalho          10:35       23:42      

In [70]:
# Labor Law Compliance Checks
from datetime import datetime, timedelta
import warnings

def time_to_minutes(time_str):
    """Convert time string HH:MM to minutes"""
    if pd.isna(time_str) or time_str is None or time_str == 'None':
        return 0
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except:
        return 0

def minutes_to_time(minutes):
    """Convert minutes to HH:MM format"""
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"

def check_labor_compliance(daily_df):
    """Check labor law compliance for working days"""
    
    # Filter only working days
    working_days = daily_df[daily_df['tipo'] == 'Trabalho'].copy().reset_index(drop=True)
    
    compliance_issues = []
    
    # Variables for period totals
    total_working_minutes = 0
    total_meal_break_minutes = 0
    total_rest_hours = 0
    rest_periods_count = 0
    
    print("=== CRONOANÁLISE DA JORNADA DE TRABALHO ===\n")
    
    # Check 1: Daily working hours > 8 hours
    print("1. VALIDAÇÃO JORNADA DE TRABALHO DE 8 HORAS")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        jornada_minutes = time_to_minutes(row['jornada_diaria'])
        jornada_hours = jornada_minutes / 60
        
        # Add to total working hours
        total_working_minutes += jornada_minutes
        
        if jornada_minutes > 480:  # 8 hours = 480 minutes
            excess_minutes = jornada_minutes - 480
            excess_time = minutes_to_time(excess_minutes)
            print(f"⚠️ {row['data']}: {row['jornada_diaria']} (EXCESO DE {excess_time} HORAS)")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Jornada diária excessiva',
                'details': f"Jornada trabalhada {row['jornada_diaria']}, excesso: {excess_time}"
            })
        else:
            print(f"✅ {row['data']}: {row['jornada_diaria']}")
    
    # Total for working hours section
    total_working_time = minutes_to_time(total_working_minutes)
    print(f"\n📊 TOTAL DE HORAS TRABALHADAS NO PERÍODO: {total_working_time}")
    
    # Check 2: Meal break >= 1 hour
    print(f"\n2. VALIDAÇÃO DO INTERVALO DE REFEIÇÃO (>= 1 hora)")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        refeicao_minutes = time_to_minutes(row['total_refeicao'])
        
        # Add to total meal break minutes
        total_meal_break_minutes += refeicao_minutes
        
        if refeicao_minutes < 60:  # Less than 1 hour
            refeicao_time = minutes_to_time(refeicao_minutes) if refeicao_minutes > 0 else "Sem registro"
            print(f"⚠️ {row['data']}: {refeicao_time}")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Intervalo de refeição insuficiente',
                'details': f"Intervalo de refeição: {refeicao_time}, necessário: 01:00"
            })
        else:
            print(f"✅ {row['data']}: {row['total_refeicao']}")
    
    # Total for meal break section
    total_meal_break_time = minutes_to_time(total_meal_break_minutes)
    print(f"\n🍽️ TOTAL DE HORAS DE INTERVALO DE REFEIÇÃO NO PERÍODO: {total_meal_break_time}")
    
    # Check 3: Rest period between shifts >= 11 hours
    print(f"\n3. VALIDAÇÃO DO PERÍODO DE DESCANSO ENTRE TURNOS (>= 11 horas)")
    print("-" * 50)
    
    for i in range(len(working_days) - 1):
        current_day = working_days.iloc[i]
        next_day = working_days.iloc[i + 1]
        
        # Parse dates and times
        try:
            current_date_str = current_day['data'].split()[0]  # Get DD/MM/YY part
            next_date_str = next_day['data'].split()[0]

            current_fim = current_day['jornada_fim']
            next_inicio = next_day['jornada_inicio']

            if pd.notna(current_fim) and pd.notna(next_inicio):
                # Convert to datetime objects for calculation
                current_date = datetime.strptime(current_date_str, '%d/%m/%y')
                next_date = datetime.strptime(next_date_str, '%d/%m/%y')
                
                # Handle end time that goes to next day (e.g., 01:26 means 01:26 next day)
                fim_hour, fim_min = map(int, str(current_fim).split(':'))
                inicio_hour, inicio_min = map(int, str(next_inicio).split(':'))
                
                # If fim time is small (like 01:26), it likely means next day
                if fim_hour < 6:  # Assuming work doesn't normally end before 6 AM
                    fim_datetime = current_date + timedelta(days=1, hours=fim_hour, minutes=fim_min)
                else:
                    fim_datetime = current_date + timedelta(hours=fim_hour, minutes=fim_min)
                
                inicio_datetime = next_date + timedelta(hours=inicio_hour, minutes=inicio_min)
                
                # Calculate rest period
                rest_period = inicio_datetime - fim_datetime
                rest_hours = rest_period.total_seconds() / 3600
                
                # Add to total rest hours
                total_rest_hours += rest_hours
                rest_periods_count += 1
                
                if rest_hours < 11:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    faltando_horas = 11 - rest_hours
                    faltando_horas_str = f"{int(faltando_horas):02d}:{int((faltando_horas % 1) * 60):02d}"
                    print(f"⚠️ {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): DESCANSO {rest_time_str} - FALTARAM {faltando_horas_str}")
                    compliance_issues.append({
                        'data': f"{current_day['data']} → {next_day['data']}",
                        'issue': 'Período de descanso insuficiente',
                        'details': f"Período de descanso: {rest_time_str}, necessário: 11:00"
                    })
                else:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    print(f"✅ {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): DESCANSO {rest_time_str}")
                    
        except Exception as e:
            print(f"⚠️  Erro calculando período de descanso entre {current_day['data']} e {next_day['data']}: {str(e)}")

    # Total for rest period section
    if rest_periods_count > 0:
        average_rest_hours = total_rest_hours / rest_periods_count
        average_rest_time = f"{int(average_rest_hours):02d}:{int((average_rest_hours % 1) * 60):02d}"
        print(f"\n😴 TEMPO MÉDIO DE DESCANSO ENTRE TURNOS: {average_rest_time}")
        print(f"🔄 NÚMERO DE PERÍODOS DE DESCANSO ANALISADOS: {rest_periods_count}")

    # Summary
    print(f"\n=== RESUMO DE CONFORMIDADE ===")
    print(f"Total de dias trabalhados analisados: {len(working_days)}")
    print(f"Total de problemas de conformidade encontrados: {len(compliance_issues)}")
    
    print(f"\n=== TOTAIS CONSOLIDADOS DO PERÍODO ===")
    print(f"📊 Total de horas trabalhadas: {total_working_time}")
    print(f"🍽️  Total de horas de intervalo de refeição: {total_meal_break_time}")
    if rest_periods_count > 0:
        print(f"😴 Tempo médio de descanso entre turnos: {average_rest_time}")
        print(f"🔄 Número de períodos de descanso analisados: {rest_periods_count}")

    if compliance_issues:
        print(f"\n=== PROBLEMAS DETALHADOS ===")
        
        # Group issues by category
        jornada_issues = [issue for issue in compliance_issues if 'Jornada diária excessiva' in issue['issue']]
        refeicao_issues = [issue for issue in compliance_issues if 'Intervalo de refeição' in issue['issue']]
        descanso_issues = [issue for issue in compliance_issues if 'Período de descanso' in issue['issue']]
        
        # Display Jornada issues
        if jornada_issues:
            print(f"\n🕐 JORNADAS DIÁRIAS EXCESSIVAS:")
            print("-" * 40)
            for issue in jornada_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
        
        # Display Meal break issues
        if refeicao_issues:
            print(f"\n🍽️  INTERVALOS DE REFEIÇÃO INSUFICIENTES:")
            print("-" * 40)
            for issue in refeicao_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
        
        # Display Rest period issues
        if descanso_issues:
            print(f"\n😴 PERÍODOS DE DESCANSO INSUFICIENTES:")
            print("-" * 40)
            for issue in descanso_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
    else:
        print("🎉 Nenhum problema de conformidade encontrado!")
    
    return compliance_issues

# Run the compliance check
compliance_issues = check_labor_compliance(df_timesheet_renamed)

=== CRONOANÁLISE DA JORNADA DE TRABALHO ===

1. VALIDAÇÃO JORNADA DE TRABALHO DE 8 HORAS
--------------------------------------------------
⚠️ 21/05/25 qua: 16:59 (EXCESO DE 08:59 HORAS)
⚠️ 22/05/25 qui: 11:58 (EXCESO DE 03:58 HORAS)
⚠️ 23/05/25 sex: 10:56 (EXCESO DE 02:56 HORAS)
⚠️ 24/05/25 sáb: 08:33 (EXCESO DE 00:33 HORAS)
⚠️ 25/05/25 dom: 10:08 (EXCESO DE 02:08 HORAS)
⚠️ 26/05/25 seg: 14:21 (EXCESO DE 06:21 HORAS)
⚠️ 27/05/25 ter: 10:31 (EXCESO DE 02:31 HORAS)
⚠️ 28/05/25 qua: 14:33 (EXCESO DE 06:33 HORAS)
⚠️ 29/05/25 qui: 08:17 (EXCESO DE 00:17 HORAS)
⚠️ 30/05/25 sex: 12:18 (EXCESO DE 04:18 HORAS)
⚠️ 31/05/25 sáb: 09:39 (EXCESO DE 01:39 HORAS)
⚠️ 01/06/25 dom: 08:53 (EXCESO DE 00:53 HORAS)
⚠️ 02/06/25 seg: 13:51 (EXCESO DE 05:51 HORAS)
⚠️ 03/06/25 ter: 11:19 (EXCESO DE 03:19 HORAS)
⚠️ 04/06/25 qua: 08:48 (EXCESO DE 00:48 HORAS)
⚠️ 05/06/25 qui: 10:10 (EXCESO DE 02:10 HORAS)
✅ 06/06/25 sex: 01:15
⚠️ 09/06/25 seg: 13:22 (EXCESO DE 05:22 HORAS)
⚠️ 10/06/25 ter: 09:14 (EXCESO DE 01:14